# MISPR Workshop — DFT & MD Workflows
### Stony Brook University · MolMD Group

Three tutorials, run entirely in Google Colab. **You need only a Google account** — no installs, no licenses, no payment.

| Section | Tutorial | Result you'll see |
|---|---|---|
| 1 | **ESP** — electrostatic partial charges (DFT) | atom charges for water + 3 solvents |
| 2 | **Binding energy** — CO₂ + propionitrile (DFT) | binding energy in kcal/mol |
| 3 | **MD** — electrolyte simulation | density plot, diffusion table, RDF curve |

The tutorials replay pre-computed Gaussian/LAMMPS outputs (`run_fake_gaussian` / `run_fake_lammps`), so everything runs in seconds on a free CPU runtime.

**How to use:** *Runtime → Run all*, wait ~2 min for setup, then watch each section produce its results. You can also run one section at a time.


## Setup — run this first (~2 min)

In [ ]:
#@title ▶ Set up the workshop environment { display-mode: "form" }
# ============================================================================
GITHUB_REPO = "https://github.com/kuldeepsinhraj/mispr-workshop-colab"
ENV_URL  = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/mispr-env.tar.gz"
DATA_URL = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/data.zip"
# ============================================================================
import os, time, shutil, subprocess, glob
CONDA, CAPSULE = "/opt/conda", "/root/capsule"
def _sh(c): subprocess.run(c, shell=True, check=True)

t0 = time.time()
os.makedirs(CAPSULE, exist_ok=True)

# 1. environment (restored to the same /opt/conda prefix -> no relocation)
print("[1/4] Downloading the Python 3.8 environment (~1.8 GB)...", flush=True)
if not os.path.exists(CONDA + "/bin/python"):
    _sh("wget -q -O /tmp/env.tar.gz '%s'" % ENV_URL)
    _sh("tar -xzf /tmp/env.tar.gz -C /")
    subprocess.run(CONDA + "/bin/conda-unpack", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _sh("rm -f /tmp/env.tar.gz")

# 2. notebooks (code/) + reference data (data/), tolerant of how data.zip is nested
print("[2/4] Fetching code and reference data...", flush=True)
if not os.path.exists(CAPSULE + "/code"):
    _sh("git clone --depth 1 -q '%s' /tmp/repo" % GITHUB_REPO)
    _sh("cp -r /tmp/repo/code %s/code" % CAPSULE)
if not os.path.isdir(CAPSULE + "/data/be"):
    shutil.rmtree("/tmp/dz", ignore_errors=True)
    fname = DATA_URL.split("/")[-1]
    _sh("wget -q -O /tmp/%s '%s'" % (fname, DATA_URL))
    shutil.unpack_archive("/tmp/" + fname, "/tmp/dz")
    # find the folder that actually contains be/ esp/ md/ (handles data/ wrapper or not)
    root = None
    for r, d, f in os.walk("/tmp/dz"):
        if all(os.path.isdir(os.path.join(r, x)) for x in ["be", "esp", "md"]):
            root = r; break
    if not root:
        raise SystemExit("data.zip does not contain be/ esp/ md/ — re-export from Code Ocean.")
    os.makedirs(CAPSULE + "/data", exist_ok=True)
    for x in ["be", "esp", "md"]:
        dst = CAPSULE + "/data/" + x
        if not os.path.exists(dst):
            shutil.copytree(os.path.join(root, x), dst)
os.makedirs(CAPSULE + "/results", exist_ok=True)

# verify the reference dirs the fake runners replay
need = [CAPSULE+"/data/esp/water/ESP", CAPSULE+"/data/be/co2/Optimization",
        CAPSULE+"/data/md/runs", CAPSULE+"/data/md/result/npt/log.lammps"]
miss = [p for p in need if not os.path.exists(p)]
if miss:
    raise SystemExit("Missing reference data:\n  " + "\n  ".join(miss))

# 3. local MongoDB from the packed env (offline)
print("[3/4] Starting MongoDB...", flush=True)
os.makedirs(CAPSULE + "/db", exist_ok=True)
menv = os.environ.copy(); menv["LD_LIBRARY_PATH"] = CONDA + "/lib:" + menv.get("LD_LIBRARY_PATH","")
subprocess.run([CONDA+"/bin/mongod","--dbpath",CAPSULE+"/db","--bind_ip","127.0.0.1",
                "--port","27017","--logpath",CAPSULE+"/db/mongod.log","--fork"],
               env=menv, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    if subprocess.run("ss -ltn 2>/dev/null | grep -q 27017", shell=True).returncode == 0: break
    time.sleep(1)
else:
    raise SystemExit("MongoDB failed to start — see " + CAPSULE + "/db/mongod.log")

# 4. helpers used by the tutorial sections
print("[4/4] Preparing helpers...", flush=True)
def run_in_env(script):
    """Run tutorial code with the packed Python 3.8 env; stream output live."""
    open("/tmp/_cell.py","w").write(script)
    e = os.environ.copy()
    e["PATH"] = CONDA + "/bin:" + e["PATH"]
    e["LD_LIBRARY_PATH"] = CONDA + "/lib:" + e.get("LD_LIBRARY_PATH","")
    e["FW_CONFIG_FILE"] = CAPSULE + "/code/config/FW_config.yaml"
    p = subprocess.Popen([CONDA+"/bin/python","-u","/tmp/_cell.py"], env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait()
    if p.returncode: print("\n[the step above exited with an error]")

def show_img(path, width=None, height=None):
    from IPython.display import Image, display
    if os.path.exists(path): display(Image(path, width=width, height=height))
    else: print("(image not found: %s)" % path)

_QUIET = "import logging,warnings; warnings.filterwarnings('ignore'); logging.getLogger('fireworks').setLevel(logging.ERROR)\n"

print("\n✅ Ready in %ds. Run the three sections below." % int(time.time()-t0))


---
## 1 · Electrostatic partial charges (ESP)

A DFT workflow that computes the net atomic (partial) charge on each atom. We feed in pre-computed Gaussian output and ask MISPR to run the ESP step, first for **water**, then for **acetonitrile, cyclohexane, and DMSO**.

In [ ]:
#@title Run the ESP workflow (water + 3 solvents)
esp_script = _QUIET + r"""
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.esp import get_esp_charges
from fireworks import LaunchPad
from fireworks.core.rocket_launcher import rapidfire
from datetime import date
import json

lpad = LaunchPad.auto_load()
lpad.reset(date.today().strftime('%Y-%m-%d'))

# --- water ---
wf, _ = get_esp_charges(
    mol_operation_type='get_from_gout_file',
    mol='/root/capsule/data/esp/water/water.out',
    process_mol_func=False, mol_name='water', skips=['opt','freq'],
    esp_gaussian_inputs={'route_parameters': {'Density': None}},
    save_to_db=True, save_to_file=True, working_dir='/root/capsule/results/esp')
wf = run_fake_gaussian(wf, ref_dirs=['/root/capsule/data/esp/water/ESP'],
                       input_files=['water_esp.com'], tolerance=10)
lpad.add_wf(wf)
rapidfire(lpad)
with open('/root/capsule/results/esp/water/analysis/esp.json') as f:
    print('\n water \n', json.load(f)['esp'])

# --- acn, cyclohexane, dmso ---
esp_inputs = {'functional':'B3LYP','basis_set':'6-31+G*',
    'route_parameters':{'pop':'MK','iop(6/50=1)':None,'NoSymmetry':None,'Density':None,'test':None},
    'link0_parameters':{'%chk':'esp.chk','%mem':'45GB','%NProcShared':'28'}}
for mol in ['acn','cyclohexane','dmso']:
    wf, _ = get_esp_charges(
        mol_operation_type='get_from_gout_file',
        mol=f'/root/capsule/data/esp/{mol}/{mol}.out',
        process_mol_func=False, mol_name=mol, skips=['opt','freq'],
        esp_gaussian_inputs=esp_inputs,
        save_to_db=True, save_to_file=True, working_dir='/root/capsule/results/esp')
    wf = run_fake_gaussian(wf, ref_dirs=[f'/root/capsule/data/esp/{mol}/ESP'],
                           input_files=[f'{mol}_esp.com'], tolerance=10)
    lpad.add_wf(wf)
rapidfire(lpad)
for mol in ['acn','cyclohexane','dmso']:
    with open(f'/root/capsule/results/esp/{mol}/analysis/esp.json') as f:
        print('\n', mol, '\n', json.load(f)['esp'])
print('\n=== ESP DONE ===')
"""
run_in_env(esp_script)


In [ ]:
#@title Charge distribution on water (red = negative O, blue = positive H)
show_img("/root/capsule/data/esp/water/esp_water.png", width=300)
show_img("/root/capsule/data/esp/esp.png", width=600)


---
## 2 · Binding energy: CO₂ + propionitrile

A 7-firework DFT workflow: optimize + frequency for each molecule and the complex, then compute the binding energy $BE = E_{final} - (E_{mol1}+E_{mol2})$.

In [ ]:
#@title Show the two input molecules
show_img("/root/capsule/data/be/Carbon_Dioxide.png", width=400)
show_img("/root/capsule/data/be/Propionitrile.png", width=500)


In [ ]:
#@title Run the binding-energy workflow
be_script = _QUIET + r"""
import os
from pymatgen.core.structure import Molecule
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.binding_energy import get_binding_energies
from fireworks import LaunchPad
from fireworks.core.rocket_launcher import rapidfire
from datetime import date
import json, numpy as np

ref_dir = '/root/capsule/data/be'
co2_mol = Molecule.from_file(ref_dir + '/co2.xyz')
propionitrile_mol = Molecule.from_file(ref_dir + '/propionitrile.xyz')
print('CO2:\n', co2_mol, '\n')
print('Propionitrile:\n', propionitrile_mol, '\n')

wf = get_binding_energies(
    mol_operation_type=['get_from_mol','get_from_mol'],
    mol=[propionitrile_mol, co2_mol], index=[3, 2], process_mol_func=False,
    save_to_db=True, save_to_file=True, mol_name=['propionitrile','co2'],
    working_dir='/root/capsule/results/be',
    opt_gaussian_inputs={'functional':'B3LYP','basis_set':'6-31+G*',
        'route_parameters':{'Opt':'(calcfc,tight,cartesian,maxcycles=100)','SCF':'(tight,xqc)',
            'int':'ultrafine','NoSymmetry':None,'test':None,'EmpiricalDispersion':'GD3','symmetry':'none'},
        'link0_parameters':{'%chk':'opt.chk','%mem':'45GB','%nprocshared':'28'}},
    freq_gaussian_inputs={'functional':'B3LYP','basis_set':'6-31+G*',
        'route_parameters':{'Freq':None,'iop(7/33=1)':None,'NoSymmetry':None,'test':None,'EmpiricalDispersion':'GD3'},
        'link0_parameters':{'%chk':'freq.chk','%mem':'45GB','%nprocshared':'28'}})

wf = run_fake_gaussian(wf,
    ref_dirs=[ref_dir+'/propionitrile/Optimization', ref_dir+'/propionitrile/Frequency',
              ref_dir+'/co2/Optimization', ref_dir+'/co2/Frequency',
              ref_dir+'/propionitrile_co2/Optimization', ref_dir+'/propionitrile_co2/Frequency'],
    input_files=['propionitrile_opt.com','propionitrile_freq.com','co2_opt.com','co2_freq.com',
                 'propionitrile_co2_opt.com','propionitrile_co2_freq.com'], tolerance=10)

lpad = LaunchPad.auto_load()
lpad.reset(date.today().strftime('%Y-%m-%d'))
lpad.add_wf(wf)
rapidfire(lpad)

with open('/root/capsule/results/be/analysis/binding_energy.json') as f:
    data = json.load(f)
be = np.round(data['be_eV']['value'] * 23, 2)
print(f'\n=== Binding energy between CO2 and Propionitrile: {be} kcal/mol ===')
"""
run_in_env(be_script)


In [ ]:
#@title The optimized complex
show_img("/root/capsule/data/be/co2_propionitrile.png", width=300)


---
## 3 · Molecular dynamics: an electrolyte

An MD workflow on a TBA⁺ / BF₄⁻ / CO₂ / propionitrile mixture: energy minimization → NPT → melt → quench → NVT, then analysis (density, diffusion, RDF).

In [ ]:
#@title System composition
show_img("/root/capsule/data/md/composition.png", width=600)
show_img("/root/capsule/data/md/mixture.png", width=500)
print("\nForce-field data file (first lines):\n")
print(open("/root/capsule/data/md/data.mixture").read(1008))


In [ ]:
#@title Run the MD workflow (min -> npt -> melt -> quench -> nvt)
md_script = _QUIET + r"""
from pymatgen.core.structure import Molecule
from mispr.lammps.utilities.utilities import run_fake_lammps
from mispr.lammps.workflows.base import lammps_workflow
from fireworks import LaunchPad
from fireworks.core.rocket_launcher import rapidfire
from datetime import date

bf4_mol = Molecule.from_file('/root/capsule/data/md/bf4.xyz'); bf4_mol.set_charge_and_spin(-1, 1)
tba_mol = Molecule.from_file('/root/capsule/data/md/tba.xyz'); tba_mol.set_charge_and_spin(1, 1)
co2_mol = Molecule.from_file('/root/capsule/data/md/co2.xyz')
propionitrile_mol = Molecule.from_file('/root/capsule/data/md/propionitrile.xyz')
print('Loaded 4 species: tba, propionitrile, bf4, co2\n')

mol_names = ['tba','propionitrile','bf4','co2']
num_mols = [20, 2625, 20, 20]
num_atoms_per_mol = [53, 9, 5, 3]
mass = [14.0067,12.0107,1.00794,14.0067,12.0107,12.0107,1.00794,10.811,18.998403,15.9994,12.0107]

wf = lammps_workflow(
    data_file_name='/root/capsule/data/md/data.mixture',
    working_dir='/root/capsule/results/md',
    recipe=[['min',['template_filename','min.in']],['npt',['template_filename','npt.in']],
            ['melt',['template_filename','melt.in']],['quench',['template_filename','quench.in']],
            ['nvt',['template_filename','nvt.in']]],
    analysis_list=['diffusion','rdf'],
    analysis_settings=[
        {'diff_names':mol_names,'timestep':1,'com_drift':True,'msd_method':'from_dump',
         'dump_freq':50000,'diff_dist':True,'avg_interval':True,
         'num_mols':num_mols,'num_atoms_per_mol':num_atoms_per_mol},
        {'rdf_type':'molecular','partial_relations':[[11,11,11],[1,2,3]],'use_default_atom_ids':True,
         'mass':mass,'num_mols':num_mols,'num_atoms_per_mol':num_atoms_per_mol}],
    template_dir='/root/capsule/data/md/templates')

wf = run_fake_lammps(wf, ref_dirs=['/root/capsule/data/md/runs/min','/root/capsule/data/md/runs/npt',
    '/root/capsule/data/md/runs/melt','/root/capsule/data/md/runs/quench','/root/capsule/data/md/runs/nvt'])

lpad = LaunchPad.auto_load()
lpad.reset(date.today().strftime('%Y-%m-%d'))
lpad.add_wf(wf)
rapidfire(lpad)
print('\n=== MD WORKFLOW DONE ===')
"""
run_in_env(md_script)


In [ ]:
#@title System visualization (NPT trajectory)
show_img("/root/capsule/data/md/npt.gif", width=500)


In [ ]:
#@title Density (generated with mdproptools)
den_script = _QUIET + r"""
from pymatgen.io.lammps.outputs import parse_lammps_log
from mdproptools.utilities.fluctuations import plot_fluctuations
log = parse_lammps_log('/root/capsule/data/md/result/npt/log.lammps')
plot_fluctuations(log[0], 'Density', 'Density', 'density.png',
                  working_dir='/root/capsule/data/md/result')
print('density.png written')
"""
run_in_env(den_script)
show_img("/root/capsule/data/md/result/density.png", width=600)
print("x-axis: seconds   ·   y-axis: density (g/cm^3)")


In [ ]:
#@title Diffusivity
import pandas as pd
df_diff = pd.read_csv("/root/capsule/data/md/result/analysis/diff/diffusion.csv", index_col=0)
df_diff.columns = ["type", "diffusion (m2/s)", "std", "R2"]
from IPython.display import display
display(df_diff)


In [ ]:
#@title Radial distribution function (C of CO2 — propionitrile)
import pandas as pd
import matplotlib.pyplot as plt
df_rdf = pd.read_csv("/root/capsule/data/md/result/analysis/rdf/molecular/0.5/rdf.csv")
plt.figure()
plt.plot(df_rdf[r"r ($\AA$)"], df_rdf["g_11-2"], label="C (CO2) - Propionitrile", color="black", linewidth=1.5)
plt.legend(fontsize=14, frameon=False)
plt.xlabel("distance (Angstroms)", fontsize=14)
plt.ylabel("RDF", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.show()


---
## (Optional) Save your results to Google Drive
Colab wipes the runtime when it disconnects. Run this to keep your outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/mispr_workshop"
!cp -r /root/capsule/results "/content/drive/MyDrive/mispr_workshop/"
print("Saved to Google Drive -> MyDrive/mispr_workshop/results")
